In [4]:
import torch
import torch.nn as nn
from torch import tensor
from torch.utils.data import Dataset, DataLoader, random_split
import mlflow
from datetime import datetime
import os
from dotenv import load_dotenv
load_dotenv(override=True)

db_uri = os.environ["ML_FLOW_TRACKING_URI"]

In [2]:
class MyDataset(Dataset):
    def __init__(self, x1, x2, x3, x4, y):
        self.x = torch.stack([x1, x2, x3, x4], dim=1)
        self.y = y.unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

n = 200

x1 = torch.rand(n) * 10
x2 = torch.rand(n) * 20
x3 = torch.rand(n) * 40
x4 = torch.rand(n) * 100

noise = torch.randn(n) * 2

y = (
    0.4 * x1
    + 0.3 * x2
    + 0.7 * x3
    + 0.3 * x4
    + 0.5 * x1 * x2
    + 0.2 * x3**2
    + 0.1 * torch.sin(x4)
    + noise
)

train_size = 8
val_size = 2

train_x1 = x1[:train_size]
train_x2 = x2[:train_size]
train_x3 = x3[:train_size]
train_x4 = x4[:train_size]
train_y = y[:train_size]

val_x1 = x1[train_size:]
val_x2 = x2[train_size:]
val_x3 = x3[train_size:]
val_x4 = x4[train_size:]
val_y = y[train_size:]

train_dataset = MyDataset(
    train_x1,
    train_x2,
    train_x3,
    train_x4,
    train_y,
)

val_dataset = MyDataset(
    val_x1,
    val_x2,
    val_x3,
    val_x4,
    val_y,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=True
)

In [5]:
mlflow.set_tracking_uri(db_uri)

In [6]:
gcs_bucket_url = "gs://my-model-training/mlflow-artifacts"

In [7]:
experiment_name = "Real-Model"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        name=experiment_name,
        artifact_location=gcs_bucket_url,
    )
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(experiment_id=experiment_id)

<Experiment: artifact_location='gs://my-model-training/mlflow-artifacts', creation_time=1787743123774, experiment_id='3', last_update_time=1787743123774, lifecycle_stage='active', name='Real-Model', tags={}, trace_location=None, workspace='default'>

In [9]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
custom_run_name = f"Experiment_{timestamp}"
print(custom_run_name)

with mlflow.start_run(run_name=custom_run_name):
    epochs = 20
    lr = 0.001
    dropout = 0.1
    num_hidden_features = 4

    mlflow.log_param("epochs", epochs)
    mlflow.log_param("lr", lr)
    mlflow.log_param("dropout", dropout)
    mlflow.log_param("num_hidden_features", num_hidden_features)
    
    model = nn.Sequential(
        nn.Linear(4, num_hidden_features),
        nn.Dropout(p=dropout),
        nn.Linear(num_hidden_features, 1),
    )
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )
    
    train_loss_list = []
    val_loss_list = []
    
    for epoch in range(epochs):
        model.train()
        
        train_loss = 0.0
        for batch in train_loader:
            x, y_target = batch
    
            optimizer.zero_grad()
            
            y_pred = model(x)
    
            loss = loss_fn(y_pred, y_target)
            loss.backward()
            
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        train_loss_list.append(train_loss)
    
        model.eval()
        with torch.no_grad():
            val_loss = 0.0
            for batch in val_loader:
                x, y_target = batch
                y_pred = model(x) 
                loss = loss_fn(y_pred, y_target)
                
                val_loss += loss.item()
    
            val_loss /= len(val_loader)
            val_loss_list.append(val_loss)
    
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"- train_loss: {train_loss:.4f} "
            f"- val_loss: {val_loss:.4f}"
        )
        
        mlflow.log_metric(
            "train_loss",
            train_loss,
            step=epoch
        )

        mlflow.log_metric(
            "val_loss",
            val_loss,
            step=epoch
        )
        
    torch.save(model.state_dict(), "model.pth")
    mlflow.log_artifact("model.pth")

Experiment_20260826-182517
Epoch 1/20 - train_loss: 27182.0733 - val_loss: 38179.9295
Epoch 2/20 - train_loss: 27399.8120 - val_loss: 37952.0960
Epoch 3/20 - train_loss: 26863.9021 - val_loss: 37732.2794
Epoch 4/20 - train_loss: 26727.7837 - val_loss: 37517.8585
Epoch 5/20 - train_loss: 26556.4109 - val_loss: 37290.3370
Epoch 6/20 - train_loss: 26141.5312 - val_loss: 37073.9144
Epoch 7/20 - train_loss: 25634.6431 - val_loss: 36867.0890
Epoch 8/20 - train_loss: 25648.7894 - val_loss: 36666.9655
Epoch 9/20 - train_loss: 25674.3108 - val_loss: 36456.6826
Epoch 10/20 - train_loss: 25344.3479 - val_loss: 36250.2027
Epoch 11/20 - train_loss: 25458.3260 - val_loss: 36053.3692
Epoch 12/20 - train_loss: 25401.1507 - val_loss: 35854.3134
Epoch 13/20 - train_loss: 25493.0380 - val_loss: 35661.7345
Epoch 14/20 - train_loss: 24647.5609 - val_loss: 35466.7714
Epoch 15/20 - train_loss: 24481.8157 - val_loss: 35274.4792
Epoch 16/20 - train_loss: 24336.0006 - val_loss: 35075.8757
Epoch 17/20 - train_lo

In [10]:
!mlflow ui --backend-store-uri "$db_uri"

Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/08/26 18:25:48 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/08/26 18:25:48 INFO:     Started parent process [22235]
2026/08/26 18:25:54 INFO:     Started server process [22241]
2026/08/26 18:25:54 INFO:     Waiting for application startup.
2026/08/26 18:25:54 INFO:     Application startup complete.
2026/08/26 18:25:54 INFO:     Started server process [22239]
2026/08/26 18:25:54 INFO:     Waiting for application startup.
2026/08/26 18:25:54 INFO:     Application startup complete.
2026/08/26 18:25:54 INFO:     127.0.0.1:42152 - "GET / HTTP/1.1" 200 OK
2026/08/26 18:25:54 INFO:     Started server process [22238]
2026/08/26 18:25:54 INFO:     Waiting for application startup.
2026/08/26 18:25:54 INFO:     